# Week 3, day 4 (morning) — Worksheet 01 SOLUTIONS: the operational source model

Executed in the lab image (pandas 3.0.5) against the generated source data.
Every quoted number is what it actually printed.

Question 4 is the one to re-read. The join that answers a simple business
question also changes the number of rows you are counting, and nothing in the
result says so.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 01 — The operational source model. Run this once.
import pandas as pd

DATA = "data/"

def load(name):
    """Read one source table by its name on slide 24."""
    return pd.read_csv(DATA + name + ".csv")

TABLES = ["category", "city", "cohort", "course", "discount_type",
          "employee", "employee_type", "enrollment", "payment_type",
          "program", "students", "transaction"]

print("the operational source model:", len(TABLES), "tables")

PART A — what the source looks like

### Question 1

Load every table in `TABLES` and print, for each, its name, row count, column count and column names. This is your map for the rest of the day.

In [ ]:
for name in TABLES:
    df = load(name)
    print("%-15s %5d x %2d" % (name, df.shape[0], df.shape[1]))
    print("                %s" % ", ".join(df.columns))

Twelve tables, 7,973 rows between them:

```
category            5 x  6
city               30 x  6
cohort             16 x  4
course             24 x  8
discount_type       6 x  4
employee           14 x  8
employee_type       3 x  2
enrollment       2400 x  6
payment_type        5 x  2
program             8 x  7
students          606 x 11
transaction      4856 x  8
```

Note the shape of it. **Two tables hold almost all the rows** — `enrollment`
(2,400) and `transaction` (4,856) — and the other ten are small: 3 to 606 rows,
mostly under 30.

That split is not an accident of this dataset; it is what an OLTP schema looks
like. The big tables record **events** as they happen. The small ones hold the
**things** the events refer to: courses, cohorts, cities, payment types. The
events accumulate forever; the reference tables barely grow.

Which is already the fact/dimension distinction, before anyone has said the
words. `enrollment` and `transaction` will become fact tables. Most of the rest
will become dimensions. Worksheet 03 does that properly — for now, just notice
that the operational schema has the same underlying shape, and that the
dimensional model is a rearrangement of it rather than a different set of facts.

Also worth noting: the widest table is `students` at 11 columns and the narrowest
is `employee_type` at 2. Nothing here is wide. OLTP rows are narrow because
inserts and updates touch whole rows, and a narrow row is a cheap one to write.

### Question 2

The business question is *"how many enrollments per program category?"*. Print the columns of `enrollment`, `course`, `program` and `category`, and write down the chain of joins that gets from one to the other.
> **NOTE:** `enrollment` has no `program_id` and no `category_id`. Follow the foreign keys.

In [ ]:
for name in ("enrollment", "course", "program", "category"):
    print("%-12s %s" % (name, list(load(name).columns)))
print()
print("enrollment.course_id -> course.course_id")
print("course.program_id    -> program.program_id")
print("program.category_id  -> category.category_id")

```
enrollment   ['enrl_id', 'enrl_date', 'stu_id', 'course_id', 'cohort_id', 'status']
course       ['course_id', 'course_name', 'course_desc', 'gov_code', 'full_time', 'hours', 'program_id', 'active_flg']
program      ['program_id', 'program_name', 'program_desc', 'pm_id', 'start_date', 'active_flg', 'category_id']
category     ['category_id', 'category_name', 'category_desc', 'director_id', 'start_date', 'active_flg']
```

Three hops:

```
enrollment.course_id -> course.course_id
course.program_id    -> program.program_id
program.category_id  -> category.category_id
```

`enrollment` has six columns and five of them are keys or dates. It knows the
`course_id`; it does not know the program, and it certainly does not know the
category. Every descriptive attribute lives one or more tables away.

This is **correct** for an operational system. Storing `category_name` on the
enrollment row would mean updating 2,400 rows when a category is renamed, and
risking 2,400 rows disagreeing if the update half-failed. Normalisation stores
each fact once so it can be changed once.

The cost lands on whoever asks a question. Three joins to attach one label, and
you have to know the chain before you can write the query — which means reading
the ER diagram, or knowing someone who has.

That cost is what the dimensional model is buying out. Hold the number three;
worksheet 09 makes the same attribute available with no joins at all.

### Question 3

Now answer it. Join `enrollment` to `course` to `program` to `category`, then count enrollments per `category_name`. Print the result and the number of joins it took.

In [ ]:
enr = load("enrollment")
q = (enr
     .merge(load("course")[["course_id", "program_id"]], on="course_id")
     .merge(load("program")[["program_id", "category_id"]], on="program_id")
     .merge(load("category")[["category_id", "category_name"]], on="category_id"))
print("joins needed:", 3)
print()
print(q.groupby("category_name").size().sort_values(ascending=False)
       .rename("enrollments").to_string())
print()
print("rows counted:", len(q), "of", len(enr), "enrollments")

```
joins needed: 3

category_name
Cloud Computing     616
Data Science        603
Data Engineering    587
Cybersecurity       278

rows counted: 2400 of 2400 enrollments
```

Three joins, and an answer. Cloud Computing leads with 616.

Now add the four numbers up: 616 + 603 + 587 + 278 = **2,084**. The line
underneath says the query counted **2,400** rows.

**316 enrollments are missing from a result that reports no error.** Question 4
finds them.

This is worth pausing on, because the output looks complete. Four categories,
sensible counts, sorted descending — it has every visual property of a finished
answer. The only thing wrong with it is the total, and totals are exactly what
nobody checks on a result that is already sorted and formatted.

Print the row count beside every aggregate you produce. It costs one line.

### Question 4

Add up the four numbers question 3 printed and compare the total with 2,400. Then find where the difference went: print the row count after each of the three merges, and the count of rows whose `category_name` is null.
> **NOTE:** the join is not what lost them. Check `len()` after every merge before blaming it.

In [ ]:
enr = load("enrollment")
print("start                        ", len(enr))
a = enr.merge(load("course")[["course_id", "program_id"]], on="course_id")
print("+ course                     ", len(a))
b = a.merge(load("program")[["program_id", "category_id"]], on="program_id")
print("+ program                    ", len(b))
c = b.merge(load("category")[["category_id", "category_name"]], on="category_id")
print("+ category                   ", len(c))
print()
grouped = c.groupby("category_name").size()
print("rows going into the groupby: ", len(c))
print("rows the groupby reported:   ", int(grouped.sum()))
print("rows with a null category:   ", int(c.category_name.isna().sum()))
print()
print("the category table:")
print(load("category")[["category_id", "category_name"]].to_string(index=False))

The join is innocent. Not one row is lost at any of the three merges:

```
start                         2400
+ course                      2400
+ program                     2400
+ category                    2400
```

The loss happens afterwards:

```
rows going into the groupby:  2400
rows the groupby reported:    2084
rows with a null category:    316
```

**`groupby` silently drops rows whose key is null.** `dropna=True` is the
default, so 316 enrollments went in and did not come out, without a warning.

The null comes from the source:

```
 category_id    category_name
           1 Data Engineering
           2     Data Science
           3  Cloud Computing
           4    Cybersecurity
           5              NaN
```

Category 5 exists — it has an id, a description and an active flag — but its
name was never filled in. The `Foundations Bootcamp` program points at it, and
every enrollment in that program's courses inherits the null.

Two separate lessons, and they compound.

**The behaviour:** `groupby(...).size()` is not conservative. Ordinary pandas
operations that drop rows on null keys include `groupby`, `pivot_table`, and any
`merge` on a null key. Pass `dropna=False` when you want to see them:

```python
c.groupby("category_name", dropna=False).size()
```

**The modelling decision:** a null in a dimension attribute is a problem you
solve once, at load time, not in every query. Slide 36 says exactly what to do
with it — *missing category → `Unknown`* — and that is worksheet 06. Once
`dim_program.program_category` reads `Unknown`, those 316 enrollments appear in
every report as a visible category with a visible count, and someone eventually
asks why 13% of enrollments are uncategorised. That question never gets asked
while the rows are silently absent.

That is a large part of what dimensional modelling is for: moving a correctness
problem out of every analyst's query and into one loading rule.

PART B — the grain the source actually has

### Question 5

The lecture says payment and discount data must be *summarised to the enrollment level* before loading. Show why: print the row count of `transaction`, its distinct `enrl_id` count, and the distribution of transactions per enrollment.

In [ ]:
tx = load("transaction")
print("transaction rows: ", len(tx))
print("distinct enrl_id: ", tx.enrl_id.nunique())
print("distinct trans_id:", tx.trans_id.nunique())
print()
per = tx.groupby("enrl_id").size()
print("transactions per enrollment:")
print(per.value_counts().sort_index().rename("enrollments").to_string())

```
transaction rows:  4856
distinct enrl_id:  2243
distinct trans_id: 4856
```

`trans_id` is unique, so every row is its own transaction. But only **2,243
distinct enrollments** appear across 4,856 rows — so the table is not one row per
enrollment:

```
transactions per enrollment:
1    762
2    747
3    353
4    364
5     17
```

Most enrollments pay in one or two instalments, some in three or four. The grain
of `transaction` is **one row per payment**, not one row per enrollment.

That is why slide 28 carries the footnote *"Payment and discount transactions
are summarized at the enrollment level before loading fact_enrollment"* — a
sentence that reads like housekeeping and is in fact the single most important
instruction in the ETL design. Worksheet 07 does the summarising.

The 17 enrollments with five transactions are worth a raised eyebrow. Look at
the shape: 1 and 2 are common, 3 and 4 less so, and then a small tail at 5. That
tail is not a payment plan. Worksheet 10's duplicate check finds out what it is.

**The general move:** for any table you are about to join or aggregate, run
`groupby(key).size().value_counts()` first. It takes one line and it tells you
the grain, which is the thing every subsequent decision depends on.

### Question 6

So `transaction` is not at enrollment grain. Show what that costs a naive analyst: join `enrollment` to `transaction` and print the row count before and after, plus what `SUM(full_price)` becomes.
> **NOTE:** `full_price` is the tuition for the enrollment, repeated on every one of its transaction rows.

In [ ]:
enr, tx = load("enrollment"), load("transaction")
j = enr.merge(tx, on="enrl_id")
print("enrollment rows:      ", len(enr))
print("after joining tx:     ", len(j))
print()
one_price = tx.groupby("enrl_id")["full_price"].max()
print("SUM(full_price) over the join: %15.2f" % j["full_price"].sum())
print("SUM of one price per enrollment: %13.2f" % one_price.sum())
print("overstated by a factor of %.2f" % (j["full_price"].sum() / one_price.sum()))

```
enrollment rows:       2400
after joining tx:      4856

SUM(full_price) over the join:     25073000.00
SUM of one price per enrollment:   11628000.00
overstated by a factor of 2.16
```

The join more than doubles the row count, and **`SUM(full_price)` reports 25,073,000.00
when the real figure is 11,628,000.00.**

The mechanism is plain once you see it: `full_price` is the tuition for the
*enrollment*, and it is repeated on every one of that enrollment's transaction
rows. An enrollment paying in four instalments carries its 4,800 tuition four
times, so summing the joined table counts it four times.

What makes this dangerous rather than merely wrong is that nothing looks broken.
The join succeeded. No nulls appeared. 25,073,000.00 is a plausible number for a
training provider, and it is a number that will grow steadily and behave
sensibly month over month. It is simply 2.16 times too large — and the factor is
not even constant, because it depends on how many instalments people happen to
choose.

This is the **fan-out** trap, and it is the most common way a correct-looking
revenue figure turns out to be fiction. The rule that prevents it:

> Before summing a column after a join, ask which table that column came from,
> and whether the join preserved that table's grain.

`full_price` came from `transaction`, but it *describes* an enrollment. Summing
it at transaction grain is a category error. `payment_amount` — which genuinely
varies per row — is safe to sum at this grain, and is the only money column here
that is.

Worksheet 03 gives this its proper name: an **additive** measure is additive
*at a stated grain*, and nowhere else.

### Question 7

Not every enrollment has a transaction. Count how many `enrl_id` values appear in `enrollment` but never in `transaction`, then show how an inner join and a left join disagree about the number of enrollments.

In [ ]:
enr, tx = load("enrollment"), load("transaction")
missing = set(enr.enrl_id) - set(tx.enrl_id)
print("enrollments with no transaction:", len(missing))
print()
inner = enr.merge(tx[["enrl_id"]].drop_duplicates(), on="enrl_id", how="inner")
left = enr.merge(tx[["enrl_id"]].drop_duplicates(), on="enrl_id", how="left")
print("inner join -> %d enrollments" % len(inner))
print("left  join -> %d enrollments" % len(left))

```
enrollments with no transaction: 157

inner join -> 2243 enrollments
left  join -> 2400 enrollments
```

**157 enrollments have no transaction row at all** — someone enrolled and never
paid anything, which is an ordinary thing for a business to have.

So the choice of join word decides how many enrollments your warehouse contains.
`inner` gives 2,243. `left` gives 2,400. Both are defensible, and they are
answers to different questions:

- **inner** answers "what did we get paid for" — it is a table of *payments*,
  filtered to enrollments as a side effect
- **left** answers "who enrolled" — 157 of them with a null or zero amount paid,
  which is the correct record of what happened

The fact table is `fact_enrollment`, its grain is one row per enrollment, and
the business questions on slide 25 start with *"How many students enrolled each
day?"*. So it has to be **left**, and the 157 have to arrive with
`amount_paid_to_date = 0` rather than null. Worksheet 07 builds that.

Note what an inner join would have done: silently dropped the 157 worst-paying
enrollments, then reported a full-payment rate computed over the survivors. The
metric would improve, and the improvement would be invisible.

That is the general shape of the danger. **An inner join is a filter you did not
write down.** Whenever you use one, you are asserting that rows without a match
do not belong in the answer — so state what they are and count them, rather than
letting the join dispose of them quietly.

PART C — what normalisation buys and costs

### Question 8

`city` stores `provn_name` and `cntry_name` inline. Print the number of rows in `city`, the number of distinct provinces and countries, and how many times the text `"Ontario"` is physically stored.

In [ ]:
cty = load("city")
print("city rows:          ", len(cty))
print("distinct provinces: ", cty.provn_name.nunique())
print("distinct countries: ", cty.cntry_name.nunique())
print()
print("rows storing 'Ontario':", int((cty.provn_name == "Ontario").sum()))
print()
print(cty.cntry_name.value_counts().rename("cities").to_string())

```
city rows:           30
distinct provinces:  21
distinct countries:  7

rows storing 'Ontario': 4

cntry_name
Canada            14
United States     10
United Kingdom     2
Ireland            1
Germany            1
Portugal           1
Brazil             1
```

`city` has both `provn_id` **and** `provn_name`, both `cntry_id` **and**
`cntry_name`. The text `"Ontario"` is stored 4 times; `"Canada"` is stored 14
times.

So the operational schema is **already denormalised here**, and deliberately.
Fully normalised, this would be three tables — `city`, `province`, `country` —
and `"Canada"` would exist exactly once. Someone decided that a province table
holding 21 rows was not worth the join.

That decision is the entire snowflake-versus-star argument, and it is already
being made inside the source model. Worksheet 04 takes it further and measures
what it costs.

The trade-off is real in both directions. Duplication costs storage — trivial at
30 rows, less trivial at 30 million — and it costs *consistency*: 14 rows
spelling `Canada` are 14 chances for one of them to say `Canda`. A `country`
table makes that impossible by construction.

What denormalisation buys is that `SELECT cntry_name FROM city` needs no join at
all. In a dimension table that is read constantly and written once a day, that
is usually the better trade — which is why star schemas denormalise dimensions
on purpose rather than by accident.

The thing to take away is that "normalised" is not a virtue and "denormalised"
is not a shortcut. Both are answers to *how often does this change, and how often
is it read*.

### Question 9

OLTP models keep current state with status columns. Find every column in the twelve tables whose name contains `active` or `status`, and print the table, column and its distinct values.
> **NOTE:** these are the columns Step 3 of the ELT design turns into reporting rules. Find them before you need them.

In [ ]:
for name in TABLES:
    df = load(name)
    for col in df.columns:
        if "active" in col.lower() or "status" in col.lower():
            vals = df[col].value_counts(dropna=False).to_dict()
            print("%-12s %-12s %s" % (name, col, vals))

Seven state columns across five tables:

```
category     active_flg   {1: 5}
course       active_flg   {1: 23, 0: 1}
employee     status       {'active': 14}
employee     active_flg   {1: 14}
enrollment   status       {'active': 2283, 'cancelled': 117}
program      active_flg   {1: 7, 0: 1}
students     active_flg   {1: 606}
```

Three of them carry real information: **117 cancelled enrollments**, one retired
course, one retired program. The other four are uniform — every category, every
employee and every student is active — so they cost a column and tell you
nothing today.

The 117 cancelled enrollments are the ones that matter, and they are a decision
waiting to be made rather than a defect. *Should a cancelled enrollment count as
an enrollment?* The lecture's answer is on slide 36 — *cancelled enrollment →
excluded or flagged* — and note that it offers **both**, because the right answer
depends on who is asking. Finance wants them out of revenue. Admissions wants
them in, because a cancellation is a thing that happened and a rate you want to
watch.

Worksheet 06 makes that decision explicitly and writes it down. What matters
here is that the decision exists at all, and that the source model presents it as
a column rather than a question.

Notice also the redundancy in `employee`: both `status` and `active_flg`, saying
the same thing in two encodings. Nothing enforces that they agree. Two columns
for one fact is how a table ends up with rows where `status = 'active'` and
`active_flg = 0`, and then two reports that disagree.

**Find these columns before you model, not after.** They are where reporting
rules come from, and every one of them is a question for the business rather
than a decision for the engineer.

### Question 10

Finally, try the shortcut: join `enrollment` straight to `program` with `enr.merge(prg, on="program_id")`. **This is supposed to fail.** Read the error and say what it tells you about the source model.

In [ ]:
enr, prg = load("enrollment"), load("program")
print("enrollment columns:", list(enr.columns))
print("program columns:   ", list(prg.columns))
print()
print(enr.merge(prg, on="program_id").shape)

```
enrollment columns: ['enrl_id', 'enrl_date', 'stu_id', 'course_id', 'cohort_id', 'status']
program columns:    ['program_id', 'program_name', 'program_desc', 'pm_id', 'start_date', 'active_flg', 'category_id']

KeyError: 'program_id'
```

There is no shortcut. `enrollment` does not have `program_id`, so the merge has
nothing to join on and pandas says so immediately.

The error is the friendly outcome, and it is worth appreciating why. The
alternative — a system that joins anyway on whatever looks similar — would give
you a wrong answer instead of a stack trace. Here the schema simply does not
permit the question to be asked directly.

Which is the summary of this worksheet. The operational model is correct,
consistent, and normalised, and every one of these was true at the same time:

- three joins to attach one label to an enrollment (Q2, Q3)
- an aggregate that silently dropped **316 of 2,400** rows on a null key (Q4)
- a transaction table at payment grain, not enrollment grain (Q5)
- `SUM(full_price)` overstated by **2.16x** after an ordinary join (Q6)
- **157 enrollments** that an inner join would have deleted from the answer (Q7)
- reporting rules hiding in seven `status` and `active_flg` columns (Q9)

None of those are bugs in the source. They are the standing cost of asking
analytical questions of a schema designed for transactions — paid again by every
analyst, every query, forever.

A dimensional model pays that cost **once**, at load time, and writes the answer
down in a table. The grain is stated instead of inferred. The category label is
already attached and already says `Unknown` where it is missing. Payments are
already summarised to the enrollment. The reporting rule for cancellations is
already applied, and recorded somewhere you can read it.

That is what the next nine worksheets build. Worksheet 02 starts where the
lecture says every dimensional model starts: **the grain**.